# Notebook 2 — MLflow LLM Evaluation

This notebook demonstrates **MLflow's `mlflow.evaluate()`** API for scoring LLM outputs.

**What you'll learn:**
- Built-in metrics: `toxicity`, `flesch_kincaid_grade_level`, `ari_grade_level`
- Custom string-match metrics
- LLM-judge scoring (relevance, faithfulness) using a local Ollama model as judge
- Comparing evaluation results across multiple prompt strategies

**Prerequisites:** Docker stack running + `uv sync` completed

In [ ]:
import sys
sys.path.insert(0, '../src')

import mlflow
import pandas as pd
from agents.config import configure_mlflow, get_ollama_base_url, get_ollama_model

configure_mlflow(experiment_name='nb-02-evaluation')
mlflow.langchain.autolog()

print('MLflow version:', mlflow.__version__)

## 1. Prepare an Evaluation Dataset

A simple Q&A dataset with ground-truth answers.

In [ ]:
eval_data = pd.DataFrame({
    'inputs': [
        'What is MLflow?',
        'What does LangGraph add over LangChain?',
        'Why use Ollama for local LLM development?',
        'What is a model registry?',
    ],
    'ground_truth': [
        'MLflow is an open-source platform for managing the ML lifecycle.',
        'LangGraph adds stateful, cyclical multi-agent graph execution on top of LangChain.',
        'Ollama runs LLMs locally, which is free, private, and works offline.',
        'A model registry is a centralised store that tracks model versions and their lifecycle stages.',
    ],
})

eval_data.head()

## 2. Generate Model Outputs

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

llm = ChatOllama(model=get_ollama_model(), base_url=get_ollama_base_url())

def generate_answers(questions: list[str]) -> list[str]:
    answers = []
    for q in questions:
        resp = llm.invoke([HumanMessage(content=f'Answer in one clear sentence: {q}')])
        answers.append(resp.content.strip())
    return answers

eval_data['predictions'] = generate_answers(eval_data['inputs'].tolist())
eval_data[['inputs', 'predictions', 'ground_truth']]

## 3. Built-in Text Quality Metrics

In [ ]:
with mlflow.start_run(run_name='eval-builtin-metrics'):
    results = mlflow.evaluate(
        data=eval_data,
        targets='ground_truth',
        predictions='predictions',
        model_type='text',
        evaluators='default',
    )

print('Aggregate metrics:')
for k, v in results.metrics.items():
    print(f'  {k}: {v:.4f}')

## 4. Custom Metric — Answer Completeness

In [ ]:
from mlflow.metrics import make_metric
from mlflow.metrics.genai import EvaluationExample

def completeness_eval(predictions, targets, metrics):
    """
    Simple heuristic: prediction is 'complete' if it shares at least 1 
    key noun with the ground truth (lowercased).
    Returns a score in {0, 1}.
    """
    scores = []
    for pred, gt in zip(predictions, targets):
        pred_words = set(pred.lower().split())
        gt_words = set(gt.lower().split())
        overlap = pred_words & gt_words
        # Exclude stop words
        stop = {'a', 'an', 'the', 'is', 'are', 'for', 'of', 'and', 'to', 'in', 'that'}
        overlap -= stop
        scores.append(1.0 if overlap else 0.0)
    return scores

completeness_metric = make_metric(
    eval_fn=completeness_eval,
    greater_is_better=True,
    name='keyword_completeness',
)

with mlflow.start_run(run_name='eval-custom-metric'):
    results = mlflow.evaluate(
        data=eval_data,
        targets='ground_truth',
        predictions='predictions',
        model_type='text',
        extra_metrics=[completeness_metric],
    )

print('Custom metric results:')
results.tables['eval_results_table'][['inputs', 'predictions', 'keyword_completeness/v1/score']]

## 5. Compare Two Prompt Strategies

In [ ]:
def generate_verbose(questions):
    answers = []
    for q in questions:
        resp = llm.invoke([HumanMessage(
            content=f'Provide a detailed, thorough explanation in 3-4 sentences: {q}'
        )])
        answers.append(resp.content.strip())
    return answers

eval_data_verbose = eval_data.copy()
eval_data_verbose['predictions'] = generate_verbose(eval_data['inputs'].tolist())

# Run both strategies in separate runs so you can compare in the UI
for strategy, df in [('concise', eval_data), ('verbose', eval_data_verbose)]:
    with mlflow.start_run(run_name=f'eval-strategy-{strategy}'):
        mlflow.log_param('prompt_strategy', strategy)
        results = mlflow.evaluate(
            data=df,
            targets='ground_truth',
            predictions='predictions',
            model_type='text',
            extra_metrics=[completeness_metric],
        )
        print(f'[{strategy}] metrics: {results.metrics}')

## Summary

Go to **http://localhost:5000** → `nb-02-evaluation` experiment → select all runs → **Compare**.

Key takeaway: `mlflow.evaluate()` gives you a systematic, reproducible way to compare prompts, models, and retrieval strategies — the foundation of responsible LLM development.